In [4]:
import pandas as pd
import pyarrow.parquet as pq
import numpy as np


## Define target
We're trying to predict if a user will posts in the second week. 

In [5]:
# Define block-based targets: number of blocks initiated and received in week 2 (days 7..13)
blocks_path = "../data/user_activity/filtered/blocks.parquet"
blocks_table = pq.read_table(blocks_path)
blocks_df = blocks_table.to_pandas()

join_dates_path = "../data/user_activity/processed/join_dates.parquet"
join_dates_df = pd.read_parquet(join_dates_path)
join_dates_df = join_dates_df.rename(columns={'created_date': 'join_date'})
    
print(join_dates_df.shape)

# Normalize and dedupe join_dates (sample) to be safe
join_dates_df = join_dates_df.copy()
join_dates_df['did_id'] = join_dates_df['did_id'].astype(str)
join_dates_df = join_dates_df.drop_duplicates(subset='did_id').reset_index(drop=True)

# Build target_table from the canonical sample (all sampled users preserved)
target_table = pd.DataFrame({'did_id': join_dates_df['did_id'].tolist()})
print('target_table initial rows (should equal sample size):', len(target_table))

# Normalize blocks_df ids to string so merges align cleanly
blocks_df = blocks_df.copy()
blocks_df['did_id'] = blocks_df['did_id'].astype(str)
subject_col = 'subject_did_id' if 'subject_did_id' in blocks_df.columns else ('subject_id' if 'subject_id' in blocks_df.columns else None)
if subject_col is not None:
    blocks_df[subject_col] = blocks_df[subject_col].astype(str)

# --- Compute week-2 initiated counts (actor side) ---
blocks_initiator = blocks_df.merge(join_dates_df[['did_id', 'join_date']], on='did_id', how='left')
blocks_initiator['created_date'] = pd.to_datetime(blocks_initiator['created_date'], utc=True, errors='coerce')
blocks_initiator['join_date'] = pd.to_datetime(blocks_initiator['join_date'], utc=True, errors='coerce')
blocks_initiator['days_since_join'] = ((blocks_initiator['created_date'] - blocks_initiator['join_date']).dt.total_seconds() / (24*3600)).round().astype('Int64')
blocks_initiator_w2 = blocks_initiator[(blocks_initiator['days_since_join'].notna()) & (blocks_initiator['days_since_join'] >= 7) & (blocks_initiator['days_since_join'] <= 13)].copy()
blocks_initiated_counts = blocks_initiator_w2.groupby('did_id').size().reset_index(name='blocks_week2_initiated')

# --- Compute week-2 received counts (subject side) ---
if subject_col is None:
    blocks_received_counts = pd.DataFrame(columns=['did_id', 'blocks_week2_received'])
else:
    blocks_subject = blocks_df.merge(join_dates_df.rename(columns={'did_id': subject_col, 'join_date': 'subject_join_date'}), on=subject_col, how='left')
    blocks_subject['created_date'] = pd.to_datetime(blocks_subject['created_date'], utc=True, errors='coerce')
    blocks_subject['subject_join_date'] = pd.to_datetime(blocks_subject['subject_join_date'], utc=True, errors='coerce')
    blocks_subject['days_since_join_subject'] = ((blocks_subject['created_date'] - blocks_subject['subject_join_date']).dt.total_seconds() / (24*3600)).round().astype('Int64')
    blocks_subject_w2 = blocks_subject[(blocks_subject['days_since_join_subject'].notna()) & (blocks_subject['days_since_join_subject'] >= 7) & (blocks_subject['days_since_join_subject'] <= 13)].copy()
    blocks_received_counts = blocks_subject_w2.groupby(subject_col).size().reset_index(name='blocks_week2_received')
    blocks_received_counts = blocks_received_counts.rename(columns={subject_col: 'did_id'})

# Left-merge counts into the full-sample target_table so all sample users remain present
target_table = target_table.merge(blocks_initiated_counts[['did_id', 'blocks_week2_initiated']], on='did_id', how='left')
target_table = target_table.merge(blocks_received_counts[['did_id', 'blocks_week2_received']], on='did_id', how='left')
target_table['blocks_week2_initiated'] = target_table['blocks_week2_initiated'].fillna(0).astype(int)
target_table['blocks_week2_received'] = target_table['blocks_week2_received'].fillna(0).astype(int)

print('Number of sample users with any received blocks in w2:', (target_table['blocks_week2_received']>0).sum())
print('Number of sample users with any initiated blocks in w2:', (target_table['blocks_week2_initiated']>0).sum())
print(target_table[['blocks_week2_initiated', 'blocks_week2_received']].describe())
print('Counts summary:')
print('Unique did_id in target_table:', target_table['did_id'].nunique())

(100000, 2)
target_table initial rows (should equal sample size): 100000
Number of sample users with any received blocks in w2: 2778
Number of sample users with any initiated blocks in w2: 3530
       blocks_week2_initiated  blocks_week2_received
count           100000.000000          100000.000000
mean                 0.319430               0.100730
std                  5.719203               4.221455
min                  0.000000               0.000000
25%                  0.000000               0.000000
50%                  0.000000               0.000000
75%                  0.000000               0.000000
max                739.000000            1195.000000
Counts summary:
Unique did_id in target_table: 100000


# Feature Engineering

In [5]:
# Load user activity and coerce data type.
user_path = "../data/user_activity/processed/user_activity.parquet"
user_df = pd.read_parquet(user_path)
print('Loaded user activity rows:', len(user_df))

def ensure_vec7(x):
    try:
        if pd.isna(x):
            return [0]*7
    except Exception:
        pass
    if isinstance(x, list):
        if len(x) == 7:
            return [int(v) for v in x]
        # pad or trim
        lst = [int(v) for v in x[:7]] + [0]*max(0, 7 - len(x))
        return lst
    # try converting numpy/other sequence
    try:
        lst = list(x)
        lst = [int(v) for v in lst[:7]] + [0]*max(0, 7 - len(lst))
        return lst
    except Exception:
        return [0]*7

vec_cols = ['posts_vec','blocks_actor_vec','blocks_subject_vec','follows_actor_vec','follows_subject_vec', 'likes_actor_vec','likes_subject_vec']
for c in vec_cols:
    if c not in user_df.columns:
        user_df[c] = [[0]*7 for _ in range(len(user_df))]
    else:
        user_df[c] = user_df[c].apply(ensure_vec7)

df = user_df.copy()

Loaded user activity rows: 52650


In [6]:
# Basic statistical features
# Posts features (week 1: days 0..6)
df['posts_total'] = df['posts_vec'].apply(sum)
df['posts_avg'] = df['posts_vec'].apply(lambda v: sum(v)/7.0)
df['posts_std'] = df['posts_vec'].apply(lambda v: float(np.std(v)))
df['posts_active_days'] = df['posts_vec'].apply(lambda v: sum(1 for x in v if x>0))
df['posts_day0'] = df['posts_vec'].apply(lambda v: int(v[0]))

# Blocks features (initiated vs received)
df['blocks_initiated_total'] = df['blocks_actor_vec'].apply(sum)
df['blocks_received_total'] = df['blocks_subject_vec'].apply(sum)
df['blocks_initiated_active_days'] = df['blocks_actor_vec'].apply(lambda v: sum(1 for x in v if x>0))
df['blocks_received_active_days'] = df['blocks_subject_vec'].apply(lambda v: sum(1 for x in v if x>0))

# Follows features (made vs received)
df['follows_made_total'] = df['follows_actor_vec'].apply(sum)
df['follows_received_total'] = df['follows_subject_vec'].apply(sum)
df['follows_made_active_days'] = df['follows_actor_vec'].apply(lambda v: sum(1 for x in v if x>0))
df['follows_received_active_days'] = df['follows_subject_vec'].apply(lambda v: sum(1 for x in v if x>0))

# Likes features (made vs received)
df['likes_made_total'] = df['likes_actor_vec'].apply(sum)
df['likes_received_total'] = df['likes_subject_vec'].apply(sum)
df['likes_made_active_days'] = df['likes_actor_vec'].apply(lambda v: sum(1 for x in v if x>0))
df['likes_received_active_days'] = df['likes_subject_vec'].apply(lambda v: sum(1 for x in v if x>0))

In [7]:
# Temporal / recency features: first/last active day within week 1 (0..6), -1 if none

def first_active_day(vec):
    for i, val in enumerate(vec):
        if val and val > 0:
            return i
    return -1

def last_active_day(vec):
    for i in range(len(vec)-1, -1, -1):
        if vec[i] and vec[i] > 0:
            return i
    return -1

# Posts recency
df['posts_first_active_day'] = df['posts_vec'].apply(first_active_day)
df['posts_last_active_day'] = df['posts_vec'].apply(last_active_day)

# Follows recency (made vs received)
df['follows_made_first_day'] = df['follows_actor_vec'].apply(first_active_day)
df['follows_made_last_day'] = df['follows_actor_vec'].apply(last_active_day)

# Likes recency (made vs received)
df['likes_made_first_day'] = df['likes_actor_vec'].apply(first_active_day)
df['likes_made_last_day'] = df['likes_actor_vec'].apply(last_active_day)

# Blocks recency (initiated vs received)
df['blocks_initiated_first_day'] = df['blocks_actor_vec'].apply(first_active_day)
df['blocks_initiated_last_day'] = df['blocks_actor_vec'].apply(last_active_day)

# Aggregate recency: most recent activity day across types (max of last_day values)
df['last_active_overall'] = df[['posts_last_active_day', 'follows_made_last_day', 'likes_made_last_day', 'blocks_initiated_last_day']].max(axis=1)

In [21]:
print(df)

         did_id  join_date                    posts_vec  \
0       4776298 2024-11-14        [0, 0, 0, 1, 1, 0, 0]   
1       7108668 2024-11-14   [33, 30, 7, 13, 10, 19, 6]   
2      15592324 2024-10-21  [2, 14, 46, 65, 52, 26, 18]   
3      23277214 2024-10-17   [2, 14, 7, 16, 20, 15, 30]   
4       6890000 2024-11-09       [0, 1, 0, 1, 0, 2, 18]   
...         ...        ...                          ...   
52645   9560742 2024-11-14        [0, 1, 0, 0, 0, 0, 0]   
52646   1836718 2024-11-13        [0, 0, 1, 0, 0, 0, 0]   
52647  18444813 2024-10-21        [0, 0, 0, 0, 1, 0, 0]   
52648  23589790 2024-12-24        [1, 0, 0, 0, 0, 0, 0]   
52649   3340751 2024-11-18        [2, 0, 0, 0, 0, 0, 0]   

            blocks_actor_vec     blocks_subject_vec  \
0      [0, 0, 0, 0, 0, 0, 0]  [0, 0, 0, 0, 0, 0, 0]   
1      [1, 2, 3, 1, 1, 0, 1]  [0, 0, 0, 0, 0, 0, 0]   
2      [0, 0, 0, 0, 0, 0, 0]  [0, 0, 0, 0, 0, 0, 1]   
3      [0, 0, 0, 0, 0, 0, 0]  [0, 0, 0, 0, 0, 0, 0]   
4      [0, 0, 0,

# Feature Selection

In [10]:
# Select final feature columns computed from user activity vectors
feature_columns = [
    # Basic statistics
    'posts_total', 'posts_avg', 'posts_std', 'posts_active_days', 'posts_day0',

    'blocks_initiated_total', 'blocks_received_total', 'blocks_initiated_active_days', 'blocks_received_active_days',

    'follows_made_total', 'follows_received_total', 'follows_made_active_days', 'follows_received_active_days',
    'likes_made_total', 'likes_received_total', 'likes_made_active_days', 'likes_received_active_days',
    
    # Recency features
    'posts_first_active_day', 'posts_last_active_day',
    'follows_made_first_day', 'follows_made_last_day',
    'likes_made_first_day', 'likes_made_last_day',
    'blocks_initiated_first_day', 'blocks_initiated_last_day',
    'last_active_overall',
    
]

# Ensure features exist in df (fill missing with 0)
for c in feature_columns:
    if c not in df.columns:
        df[c] = 0

# Create final feature dataset
X = df[feature_columns].fillna(0)
y = target_table['blocks_week2_initiated']

print(f"Final feature set shape: {X.shape}")
print(f"Feature columns: {list(X.columns)}")

# Check correlation with target
correlations = X.corrwith(y).sort_values(ascending=False)
print("\nTop features correlated with target:")
print(correlations.head(10))

Final feature set shape: (52650, 26)
Feature columns: ['posts_total', 'posts_avg', 'posts_std', 'posts_active_days', 'posts_day0', 'blocks_initiated_total', 'blocks_received_total', 'blocks_initiated_active_days', 'blocks_received_active_days', 'follows_made_total', 'follows_received_total', 'follows_made_active_days', 'follows_received_active_days', 'likes_made_total', 'likes_received_total', 'likes_made_active_days', 'likes_received_active_days', 'posts_first_active_day', 'posts_last_active_day', 'follows_made_first_day', 'follows_made_last_day', 'likes_made_first_day', 'likes_made_last_day', 'blocks_initiated_first_day', 'blocks_initiated_last_day', 'last_active_overall']

Top features correlated with target:
likes_made_first_day           0.008009
posts_first_active_day         0.005214
likes_received_active_days     0.001303
blocks_initiated_total         0.000650
blocks_received_total         -0.000435
likes_received_total          -0.000470
posts_last_active_day         -0.00110

# Save the file 

In [23]:
# Save features and target separately
features_output_path = "../data/user_activity/featured/features.parquet"
target_output_path = "../data/user_activity/featured/target.parquet"

# Save as separate files - Convert y to DataFrame first
X.to_parquet(features_output_path, index=False)
y.to_frame().to_parquet(target_output_path, index=False)  # Convert Series to DataFrame

print(f"\nFeatures saved to: {features_output_path}")
print(f"Target saved to: {target_output_path}")


Features saved to: ../data/posting/featured/features.parquet
Target saved to: ../data/posting/featured/target.parquet
